LangChain + OpenAI API con Python

> En LangChain v1 el punto de entrada recomendado es **`create_agent`**: un harness de modelo + tools + prompt + middleware.
> Las chains clásicas (`LLMChain`, `ConversationChain`, `ConversationBufferMemory`) viven en `langchain-classic`.
> **LCEL** (`prompt | modelo | parser`) sigue siendo útil para pipelines de un solo paso, no para orquestar una app conversacional.
> La memoria de corto plazo se resuelve con un **checkpointer** y `thread_id`, no con `RunnableWithMessageHistory` (deprecado desde 1.3.3).


## 1. Setup

Instalar dependencias. Ejecutar una vez.

Paquetes principales:

- `langchain`: agentes, mensajes, tools y `init_chat_model`.
- `langchain-openai`: integración oficial con OpenAI (se instala con el extra `[openai]`).
- `pydantic`: validación de datos y salida estructurada.

`langchain-core` y `langgraph` se instalan como dependencias. Opcional: `python-dotenv` para cargar un archivo `.env`.


In [ ]:
%pip install --upgrade "langchain[openai]" pydantic python-dotenv


## 2. API Key y configuración

No hardcodear claves en el notebook. Usar variable de entorno.

Opciones:

1. Definir `OPENAI_API_KEY` en el entorno.
2. Crear un archivo `.env` con:
   ```text
   OPENAI_API_KEY=tu_api_key
   OPENAI_MODEL=gpt-6-luna
   ```
3. Ingresarla manualmente cuando el notebook la pida.


In [12]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print(f"Modelo: {OPENAI_MODEL}")


Modelo: gpt-6-luna


## 3. Primer contacto con LangChain

En la API directa de OpenAI solemos crear un cliente y llamar a un endpoint.
En LangChain usamos una abstracción de **chat model**.

La forma recomendada de inicializarlo es `init_chat_model`. El equivalente específico de OpenAI es `ChatOpenAI`.


In [13]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    OPENAI_MODEL,
)

# Equivalente explícito:
# from langchain_openai import ChatOpenAI
# model = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)

response = model.invoke("Explicá qué es LangChain en una frase simple.")
print(response)


content='LangChain es una herramienta para crear aplicaciones que conectan modelos de inteligencia artificial con datos y otras funciones.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 18, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EZllrzS47paJsKG2VIPWRCCLa07Ct', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a14713-7055-7dd0-9391-ad2048e3b430-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 24, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 3.1 ¿Qué devuelve `invoke`?

Un modelo de chat devuelve un **mensaje**, no solo un string.

- `response.content` es el texto (o la estructura nativa del proveedor).
- `response.content_blocks` es la vista normalizada y agnóstica del proveedor: texto, reasoning, tool calls, etc.

También hay metadatos útiles, por ejemplo uso de tokens.


In [ ]:
print(response.content)
response.content_blocks


## 4. Mensajes: system, human y AI

Los modelos conversacionales trabajan con listas de mensajes.

Roles comunes:

- `system`: instrucciones generales, tono, reglas.
- `human` / `user`: mensaje del usuario.
- `ai` / `assistant`: mensaje previo del asistente.

En el paquete `langchain` se importan desde `langchain.messages`.


In [ ]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

# SystemMessage // system prompt
# HumanMessage // interacción de usuario
# AIMessage // respuesta del modelo

messages = [
    SystemMessage("Sos un profesor de programación. Respondé claro y con ejemplos cortos."),
    HumanMessage("¿Para qué sirve un agente en LangChain?"),
]

response = model.invoke(messages)
print(response.content)


## 5. Prompt templates

Un **prompt template** permite reutilizar una estructura de prompt con variables.

Esto es clave en aplicaciones reales porque separa:

- plantilla del prompt,
- datos de entrada,
- modelo,
- salida.

`ChatPromptTemplate` sigue viviendo en `langchain-core` (no está reexportado en el namespace reducido de `langchain`).


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un docente de IA para programadores. Explicá con precisión y sin humo."),
    ("human", "Explicá el concepto: {concepto}. Nivel: {nivel}. Incluí un ejemplo en Python."),
])

formatted_messages = prompt.invoke({
    "concepto": "prompt template",
    "nivel": "principiante",
})

formatted_messages


ChatPromptValue(messages=[SystemMessage(content='Sos un docente de IA para programadores. Explicá con precisión y sin humo.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explicá el concepto: prompt template. Nivel: principiante. Incluí un ejemplo en Python.', additional_kwargs={}, response_metadata={})])

In [5]:
response = model.invoke(formatted_messages)
print(response.content)


Un **prompt template** (plantilla de prompt) es un texto reutilizable con espacios para completar. Permite usar la misma estructura con distintos datos, en lugar de escribir cada prompt desde cero.

### Ejemplo en Python

```python
plantilla = "Explicá {tema} en lenguaje sencillo y con un ejemplo."

prompt = plantilla.format(tema="qué es una variable")
print(prompt)
```

El resultado es:

```text
Explicá qué es una variable en lenguaje sencillo y con un ejemplo.
```

En este caso, `{tema}` es un **marcador de posición**: se reemplaza por el valor que se pasa a `format()`.

Una plantilla por sí sola no llama a un modelo de IA. Solo construye el texto que luego podrías enviarle. Ayuda a mantener prompts consistentes y a cambiar fácilmente los datos de entrada.


## 6. LCEL: LangChain Expression Language

Para **pipelines de un solo paso** (clasificar, resumir, extraer) se puede componer con **LCEL**:

```python
chain = prompt | model | parser
```

El operador `|` conecta la salida de un paso con la entrada del siguiente.

Esto no reemplaza a un agente. Para chat con tools, memoria y control del loop, las docs actuales recomiendan `create_agent`.


In [15]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un docente de IA para programadores. Explicá con precisión y sin humo."),
    ("human", "Explicá el concepto: {concepto}. Nivel: {nivel}. Incluí un ejemplo en Python."),
])

parser = StrOutputParser()
chain = prompt | model | parser

resultado = chain.invoke({
    "concepto": "LCEL",
    "nivel": "intermedio",
})

print(resultado)


## ¿Qué es LCEL?

**LCEL** significa **LangChain Expression Language**. Es una forma de componer componentes de LangChain —prompts, modelos, parsers, retrievers y funciones— como una cadena de *Runnables*.

El operador `|` conecta componentes: la salida de uno se convierte en la entrada del siguiente. Por ejemplo:

```text
prompt | modelo | parser
```

No es un lenguaje independiente ni una garantía de que el flujo sea correcto: es una interfaz de composición dentro de Python. Los componentes deben aceptar tipos de entrada y producir tipos compatibles.

## Ejemplo en Python

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "Explicá el concepto de forma clara y breve."),
    ("human", "Tema: {tema}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = prompt | model | StrOutputParser()

respuesta = c

## 7. Parámetros del modelo

Los parámetros se definen al crear el modelo.

Parámetros útiles:

- `temperature`: controla variación/creatividad.
- `timeout`: tiempo máximo de espera.
- `max_retries`: reintentos ante errores transitorios (red, 429, 5xx).


In [ ]:
modelo_determinista = init_chat_model(
    OPENAI_MODEL,
    temperature=0,
    max_retries=2,
    timeout=30,
)

modelo_creativo = init_chat_model(
    OPENAI_MODEL,
    temperature=0.9,
    max_retries=2,
    timeout=30,
)

pregunta = "Dame 3 nombres para una app que resume PDFs con IA."

print("Determinista:")
print(modelo_determinista.invoke(pregunta).content)

print("\nCreativo:")
print(modelo_creativo.invoke(pregunta).content)


## 8. Output parsers

Un parser transforma la respuesta del modelo a un formato más cómodo.

El parser más simple es `StrOutputParser`, que extrae el texto final. Encaja bien en una chain LCEL.


In [16]:
prompt_resumen = ChatPromptTemplate.from_messages([
    ("system", "Resumí textos técnicos para estudiantes."),
    ("human", "Texto: {texto}\n\nDevolvé un resumen de 3 bullets."),
])

chain_resumen = prompt_resumen | model | StrOutputParser()

texto = '''
LangChain permite construir aplicaciones con modelos de lenguaje. El punto de entrada
recomendado es create_agent. LCEL sigue sirviendo para pipelines cortos: prompt, modelo
y parser conectados con el operador |.
'''

print(chain_resumen.invoke({"texto": texto}))


- LangChain permite crear aplicaciones con modelos de lenguaje.
- `create_agent` es el punto de entrada recomendado.
- LCEL sigue siendo útil para pipelines breves que conectan prompts, modelos y parsers con `|`.


## 9. Salida estructurada con Pydantic

Cuando una aplicación necesita datos confiables, no alcanza con pedir “respondé en JSON”.
Sobre un modelo standalone se usa `with_structured_output(...)`.

En un agente, el equivalente es `response_format=` en `create_agent`.

Ejemplo: extraer información de una descripción de curso.


In [17]:
from typing import List

from pydantic import BaseModel, Field

class Curso(BaseModel):
    titulo: str = Field(description="Título breve del curso")
    nivel: str = Field(description="Nivel: inicial, intermedio o avanzado")
    temas: List[str] = Field(description="Lista de temas principales")
    duracion_horas: float = Field(description="Duración estimada en horas")

structured_model = model.with_structured_output(Curso)

descripcion = '''
Curso intensivo para programadores que ya conocen Python y quieren aprender a crear
aplicaciones con la API de OpenAI, LangChain, RAG y agentes. Duración aproximada: 12 horas.
'''

curso = structured_model.invoke(
    f"Extraé los datos del curso desde esta descripción:\n{descripcion}"
)

curso.model_dump_json()


'{"titulo":"Aplicaciones con OpenAI, LangChain, RAG y agentes","nivel":"intermedio","temas":["API de OpenAI","LangChain","RAG","Agentes"],"duracion_horas":12.0}'

In [ ]:
print(curso.titulo)
print(curso.temas)
print(curso.model_dump())


## 10. Streaming

El streaming permite mostrar tokens a medida que se generan. Es muy útil para interfaces de chat.

Sobre el modelo, las docs actuales usan `chunk.text` (y `chunk.content_blocks` si hay reasoning o tool calls).
Una chain LCEL también tiene `.stream()`.


In [ ]:
print("Streaming del modelo:\n")
for chunk in model.stream("Explicá en 5 bullets por qué LangChain es útil para RAG."):
    print(chunk.text, end="", flush=True)

print("\n\n--- chain LCEL ---\n")

prompt_stream = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente técnico. Respondé en español."),
    ("human", "Explicá en 3 bullets qué es create_agent."),
])
chain_stream = prompt_stream | model | StrOutputParser()

for chunk in chain_stream.stream({}):
    print(chunk, end="", flush=True)


## 11. Batch

`batch` permite ejecutar la misma chain sobre muchas entradas.

Esto es útil para:

- clasificar muchos textos,
- resumir muchos documentos,
- generar evaluaciones,
- procesar filas de un dataset.


In [ ]:
prompt_clasificador = ChatPromptTemplate.from_messages([
    ("system", "Clasificá el sentimiento como Positivo, Neutral o Negativo. Respondé solo la etiqueta."),
    ("human", "Texto: {texto}"),
])

clasificador = prompt_clasificador | modelo_determinista | StrOutputParser()

entradas = [
    {"texto": "El curso fue claro y muy práctico."},
    {"texto": "No entendí nada y los ejemplos no funcionaban."},
    {"texto": "La clase empieza a las 18 hs."},
]

clasificador.batch(entradas)


## 12. Memoria conversacional

Un chat model no recuerda llamadas anteriores por sí mismo.

En tutoriales antiguos se usaba `ConversationBufferMemory` o se envolvía una chain con `RunnableWithMessageHistory`.
Esa última API está **deprecada** (desde langchain-core 1.3.3) y se va a eliminar en 2.0.

La forma actual es un **agente con checkpointer**. El historial queda en el estado del grafo, separado por `thread_id`:

1. Crear el agente con `checkpointer=InMemorySaver()` (en producción, un checkpointer de base de datos).
2. Invocar siempre con `config={"configurable": {"thread_id": "..."}}`.
3. En cada turno se manda **solo el mensaje nuevo**; el checkpointer reinyecta el historial.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tutor = create_agent(
    model=model,
    tools=[],
    system_prompt="Sos un tutor de Python y LangChain. Recordá el contexto de la conversación.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "alumno-1"}}

r1 = tutor.invoke(
    {"messages": [{"role": "user", "content": "Estoy aprendiendo LangChain y ya sé Python básico."}]},
    config=config,
)
print(r1["messages"][-1].content)

print("\n--- segunda pregunta ---\n")

r2 = tutor.invoke(
    {"messages": [{"role": "user", "content": "¿Qué debería aprender ahora? Respondé teniendo en cuenta lo que te dije antes."}]},
    config=config,
)
print(r2["messages"][-1].content)


In [ ]:
# Inspeccionar el historial persistido en el thread
estado = tutor.get_state(config)
for mensaje in estado.values["messages"]:
    print(f"{type(mensaje).__name__}: {mensaje.content}")


## 13. Tools y agentes

Un **tool** es una función que el modelo puede usar.

Un **agente** decide cuándo usar herramientas y cuándo responder directamente.

En LangChain v1 se usa `create_agent` desde `langchain.agents`. El decorator `@tool` se importa de `langchain.tools`.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def calcular_precio_con_iva(precio: float, iva: float = 0.21) -> float:
    '''Calcula el precio final sumando IVA. precio e iva deben ser números.'''
    return round(precio * (1 + iva), 2)

@tool
def recomendar_modelo(caso_de_uso: str) -> str:
    '''Recomienda una familia de solución según el caso de uso.'''
    caso = caso_de_uso.lower()
    if "rag" in caso or "document" in caso or "pdf" in caso:
        return "Usaría un modelo de chat + embeddings + retriever/vector store."
    if "clasificar" in caso or "sentimiento" in caso:
        return "Usaría un modelo de chat con salida estructurada y temperatura baja."
    return "Empezaría con un modelo de chat y evaluaría si hace falta RAG o tools."

agent = create_agent(
    model=model,
    tools=[calcular_precio_con_iva, recomendar_modelo],
    system_prompt="Sos un asistente para programadores. Usá tools cuando aporten precisión.",
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tengo un producto de 100 USD sin IVA. ¿Cuánto sale con IVA de 21%?"}
    ]
})

result


In [ ]:
# Extraer el último mensaje del agente
ultimo_mensaje = result["messages"][-1]
print(ultimo_mensaje.content)
ultimo_mensaje.content_blocks
